# Assignment 37: AstraDB RAG

# Task 1: Getting Started with AstraDB(DataStax)
## 1. Create a DataStax AstraDB Account.
## 2. Create a vector database.
## 3. Generate application token & DB endpoint.

In [1]:
import os
import cassio
from dotenv import load_dotenv

load_dotenv()

ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")


# Task 2: Connect Langchain with AstraDB
## 1. Install AStraDB Langchain integration.
## 2. connect to AstraDB using:
## . Token
## . Endpoint
## . Verify Connection

In [2]:
import os
from dotenv import load_dotenv

from langchain_astradb import AstraDBVectorStore
from langchain_ollama import OllamaEmbeddings


# 1. Load environment variables
load_dotenv()


# 2. AstraDB credentials
ASTRA_DB_APPLICATION_TOKEN = os.getenv(
    "ASTRA_DB_APPLICATION_TOKEN"
)

ASTRA_DB_API_ENDPOINT = os.getenv(
    "ASTRA_DB_API_ENDPOINT"
)


# 3. Ollama embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


# 4. Connect to AstraDB
vectorstore = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="QA_demo_one",
    token=ASTRA_DB_APPLICATION_TOKEN,
    api_endpoint=ASTRA_DB_API_ENDPOINT
)


# 5. Verify connection
vectorstore.add_texts([
    "Python is a programming language."
])

results = vectorstore.similarity_search(
    "What is Python?",
    k=1
)


# 6. Print result
print(" AstraDB connection successful!")
print(results[0].page_content)

c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 AstraDB connection successful!
Python is a programming language.


# Task 3: Load and Split PDF Document
## 1. Load a PDF using Langchain PDF loader.
## 2. Split text into chunks.
## 3. Prepare documents for embedding.

In [3]:
## LOding PDF
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample_summarization_document.pdf")

documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 2


C:\Users\Anish Rawat\AppData\Local\Temp\ipykernel_16892\324170062.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
## Split text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)


chunks = text_splitter.split_documents(documents)
print("Number of chunks:", len(chunks))

Number of chunks: 8


In [5]:
## Prepare chunks for embeddings
print(chunks[0].page_content)

Introduction to Artificial Intelligence
Sample Document for Text Summarization with LangChain
1. What is Artificial Intelligence?
Artificial Intelligence (AI) is a branch of computer science focused on building systems that can
perform tasks that normally require human intelligence. These tasks include understanding language,
recognizing images, making predictions, solving problems, and learning from data. AI systems can be
simple rule-based programs or complex machine learning models trained on large datasets.
2. Machine Learning
Machine learning is a major part of artificial intelligence. Instead of explicitly programming every rule,
developers provide data from which a model can learn patterns. Supervised learning uses labeled


# Task 4: Store Embeddings in AstraDB
## 1. Use embedding model
## 2. Store document embeddings in AstraDB vector store.
## 3. Verify data persistence.

In [18]:


# 3. Ollama embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


In [10]:
# store documents embedding into AstraDB vector store.

vectorstore = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="QA_demo_one",
    token=ASTRA_DB_APPLICATION_TOKEN,
    api_endpoint=ASTRA_DB_API_ENDPOINT
)

In [23]:
## Verify data persistence
vectorstore.add_documents(chunks)
results = vectorstore.similarity_search(
    "What is this document about?",
    k=3
)

print("Retrieved documents:", len(results))

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1}")
    print(doc.page_content[:500])


Retrieved documents: 3

--- Result 1
Introduction to Artificial Intelligence
Sample Document for Text Summarization with LangChain
1. What is Artificial Intelligence?
Artificial Intelligence (AI) is a branch of computer science focused on building systems that can
perform tasks that normally require human intelligence. These tasks include understanding language,
recognizing images, making predictions, solving problems, and learning from data. AI systems can be
simple rule-based programs or complex machine learning models trained on

--- Result 2
Introduction to Artificial Intelligence
Sample Document for Text Summarization with LangChain
1. What is Artificial Intelligence?
Artificial Intelligence (AI) is a branch of computer science focused on building systems that can
perform tasks that normally require human intelligence. These tasks include understanding language,
recognizing images, making predictions, solving problems, and learning from data. AI systems can be
simple rule-based pr

# Task 5: PDF Query RAG Application
## Build a RAG Pipeline:
## PDF--> Splitter--> Embeddings---> AstraDB--> Retriever--> LLM--> Answer

## 1. Accept user questions.
## 2.Retrieve relevent PDF chunks
## 3. Generate grounded answers

In [28]:
## Load PDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_astradb import AstraDBVectorStore
from langchain_ollama import OllamaLLM
from dotenv import load_dotenv

load_dotenv()

loader = PyPDFLoader("sample_summarization_document.pdf")

documents = loader.load()

## Split the pdf into chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

## Embeddings
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

## AStraDB 

vectorstore = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="QA_demo_one",
    token=ASTRA_DB_APPLICATION_TOKEN,
    api_endpoint=ASTRA_DB_API_ENDPOINT
)
vectorstore.add_documents(chunks)


## Retriever

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

# Ollama LLM

llm = OllamaLLM(
    model="llama3.2"
)



query = input("\nAsk a question about the PDF ")

# Retrieve relevant chunks

docs = retriever.invoke(query)


# Create Context

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

prompt = f"""
You are a question-answering assistant.

Answer the user's question ONLY using the
provided PDF context.

If the answer cannot be found in the context,
say:

"I could not find the answer in the PDF."

Do not make up information.

PDF Context:
{context}

Question:
{query}

Answer:
"""
# Generate Answer
answer = llm.invoke(prompt)
print(answer)


Artificial Intelligence (AI) is a branch of computer science focused on building systems that can perform tasks that normally require human intelligence. These tasks include understanding language, recognizing images, making predictions, solving problems, and learning from data. AI systems can be simple rule-based programs or complex machine learning models trained on large datasets.


# Task 6: Testing and Validation
## 1. Ask atleast 5 Questions from the PDF.
## 2. Verify answers come from document content.
## 3. Handle out-of-context questions gracefully.

In [29]:
## Asking 5 questions 
test_questions = [
    "What is the main topic of the document?",
    "What are the key points discussed in the document?",
    "What is the main purpose of the document?",
    "What important information is mentioned in the document?",
    "What conclusion does the document provide?"
]

for question in test_questions:
    print("QUESTION:", question)

    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Create context
    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    # Grounded prompt
    prompt = f"""
You are a document question-answering assistant.

Answer the question ONLY using the provided PDF context.

If the answer is not present in the context,
say exactly:

"I could not find the answer in the PDF."

Do not use outside knowledge.
Do not make up information.

PDF Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
    answer = llm.invoke(prompt)

    print("\nANSWER:")
    print(answer)

QUESTION: What is the main topic of the document?

ANSWER:
The main topic of the document is support, finance, healthcare research, marketing, and document analysis.
QUESTION: What are the key points discussed in the document?

ANSWER:
The key points discussed in the document include:

1. The importance of Artificial Intelligence, machine learning, generative AI, and retrieval-based systems in modern software development.
2. The concept of a document summarization pipeline as a practical project for learning how language models work with real-world data.
3. The use of tools such as LangChain to combine document loading, text processing, prompts, models, and retrieval into a single system for support, finance, healthcare research, marketing, and document analysis.

These points highlight the potential applications and technologies for building systems that can summarize meeting transcripts, search internal documents, draft reports, and answer questions about product manuals, while empha

In [31]:
## Verify answers 
for i, doc in enumerate(docs):

    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(doc.page_content[:500])


--- Retrieved Chunk 1 ---
support, finance, healthcare research, marketing, and document analysis. For example, a company
can build a system that summarizes meeting transcripts, searches internal documents, drafts reports,
or answers questions about product manuals. Such systems still require testing, privacy controls,
monitoring, and human review when mistakes could have important consequences.
8. Conclusion
Artificial Intelligence, machine learning, generative AI, and retrieval-based systems are important
technologies 

--- Retrieved Chunk 2 ---
support, finance, healthcare research, marketing, and document analysis. For example, a company
can build a system that summarizes meeting transcripts, searches internal documents, drafts reports,
or answers questions about product manuals. Such systems still require testing, privacy controls,
monitoring, and human review when mistakes could have important consequences.
8. Conclusion
Artificial Intelligence, machine learning, generative AI, 

In [ ]:
## Asking out of context quesstions
question="Who is the Prime Minister of India"

    # Retrieve relevant chunks
docs = retriever.invoke(question)

    # Create context
context = "\n\n".join(
        doc.page_content
        for doc in docs
)

    # Grounded prompt
prompt = f"""
You are a document question-answering assistant.

Answer the question ONLY using the provided PDF context.

If the answer is not present in the context,
say exactly:

"I could not find the answer in the PDF."

Do not use outside knowledge.
Do not make up information.

PDF Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer
answer = llm.invoke(prompt)

print("\nANSWER:")
print(answer)

IndentationError: unexpected indent (3320445590.py, line 37)